# Statistical Testing for Data Quality: Detecting Distribution Drift in the Breast Cancer Wisconsin Dataset

**Problem statement.** As an SQA engineer moving into ML quality, one of your core jobs will be verifying that the data flowing into a production model still matches the distribution it was trained on. In this exercise you'll work with the real **Breast Cancer Wisconsin (Diagnostic)** dataset (569 samples, 30 numeric features computed from digitized images of fine needle aspirate biopsies of breast masses). You will split the dataset into a "reference" batch and a "current" batch by sorting on the `mean radius` feature to simulate the kind of covariate drift that happens when a production data source shifts over time (e.g. a new imaging device, a different patient population, a change in sample prep). You will then build a small statistical test suite — of the same kind used by real ML monitoring tools such as Evidently or whylogs — to flag which features have drifted.

**What you should produce:**
1. `describe_batches(reference, current)` — a summary statistics table (mean, std, count) for both batches, side by side, one row per feature.
2. `ks_drift_test(reference, current)` — runs a two-sample Kolmogorov–Smirnov test **per feature** and returns a DataFrame of KS statistics and p-values, one row per feature.
3. `mean_diff_confidence_interval(reference, current, confidence=0.95)` — for each feature, a normal-approximation confidence interval for the difference in means (`current.mean() - reference.mean()`), including the standard error used.
4. `flag_drifted_features(ks_results, alpha=0.05)` — applies a **Benjamini–Hochberg** false-discovery-rate correction to the per-feature KS p-values and returns the list of features flagged as drifted at the given alpha. (Running 30 independent hypothesis tests without correction would give you a high chance of false alarms — the same problem you'd hit alerting on every metric in a monitoring dashboard.)
5. A short written conclusion (2-4 sentences, in a markdown cell) on **which features actually drifted** as a result of the sorting-induced shift, and **why** — in terms of how those features correlate with `mean radius`.

Work through the starter cells below, then compare against the solution.

## Setup: imports and data loading

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer

rng = np.random.default_rng(42)

data = load_breast_cancer(as_frame=True)
df = data.frame.drop(columns=["target"])  # 30 numeric features, 569 rows
feature_names = list(df.columns)

df.shape, df.head()


### Simulate covariate drift

We sort the dataset by `mean radius` and split it so the "current" (production-like) batch
is skewed toward larger tumors than the "reference" (training-time) batch. This is a common,
realistic way drift shows up: not a random resample, but a shift correlated with one
underlying variable (e.g. a new patient population, a new scanner's calibration).

In [ ]:
df_sorted = df.sort_values("mean radius").reset_index(drop=True)

n = len(df_sorted)
split_point = int(n * 0.6)

reference = df_sorted.iloc[:split_point].reset_index(drop=True)
current = df_sorted.iloc[split_point:].reset_index(drop=True)

print(f"reference: {reference.shape}, current: {current.shape}")


## Starter scaffolding — fill in the TODOs

In [ ]:
def describe_batches(reference: pd.DataFrame, current: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame indexed by feature name with columns:
    ['ref_mean', 'ref_std', 'ref_n', 'cur_mean', 'cur_std', 'cur_n'].
    """
    # TODO: compute mean/std/count for each column in `reference` and `current`
    # and combine them into a single DataFrame indexed by feature name.
    raise NotImplementedError


In [ ]:
def ks_drift_test(reference: pd.DataFrame, current: pd.DataFrame) -> pd.DataFrame:
    """Run a two-sample Kolmogorov-Smirnov test for each feature.

    Return a DataFrame indexed by feature name with columns ['ks_stat', 'p_value'].
    Hint: scipy.stats.ks_2samp(sample1, sample2)
    """
    # TODO: for each column, run stats.ks_2samp(reference[col], current[col])
    # and collect the statistic and p-value.
    raise NotImplementedError


In [ ]:
def mean_diff_confidence_interval(reference: pd.DataFrame, current: pd.DataFrame,
                                   confidence: float = 0.95) -> pd.DataFrame:
    """For each feature, compute a normal-approximation CI for (current.mean() - reference.mean()).

    Return a DataFrame indexed by feature name with columns:
    ['mean_diff', 'std_err', 'ci_low', 'ci_high'].

    Hint: standard error of a difference of two independent sample means is
    sqrt(s1^2/n1 + s2^2/n2). Use stats.norm.ppf for the critical value.
    """
    # TODO: implement the per-feature confidence interval for the difference in means.
    raise NotImplementedError


In [ ]:
def flag_drifted_features(ks_results: pd.DataFrame, alpha: float = 0.05) -> list:
    """Apply Benjamini-Hochberg FDR correction to ks_results['p_value'] and return
    the list of feature names flagged as drifted at the given alpha.

    Benjamini-Hochberg procedure (recap):
    1. Sort the m p-values ascending: p_(1) <= p_(2) <= ... <= p_(m).
    2. Find the largest k such that p_(k) <= (k / m) * alpha.
    3. Reject (flag) all hypotheses with rank <= k.
    """
    # TODO: implement the BH step-up procedure over ks_results sorted by p_value.
    raise NotImplementedError


In [ ]:
# TODO: call your four functions on `reference` and `current` and inspect the results.
# summary = describe_batches(reference, current)
# ks_results = ks_drift_test(reference, current)
# ci_results = mean_diff_confidence_interval(reference, current)
# drifted = flag_drifted_features(ks_results)
# print(drifted)


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def describe_batches(reference: pd.DataFrame, current: pd.DataFrame) -> pd.DataFrame:
    ref_stats = reference.agg(["mean", "std", "count"]).T
    ref_stats.columns = ["ref_mean", "ref_std", "ref_n"]
    cur_stats = current.agg(["mean", "std", "count"]).T
    cur_stats.columns = ["cur_mean", "cur_std", "cur_n"]
    return ref_stats.join(cur_stats)


def ks_drift_test(reference: pd.DataFrame, current: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in reference.columns:
        ks_stat, p_value = stats.ks_2samp(reference[col], current[col])
        rows.append({"feature": col, "ks_stat": ks_stat, "p_value": p_value})
    return pd.DataFrame(rows).set_index("feature")


def mean_diff_confidence_interval(reference: pd.DataFrame, current: pd.DataFrame,
                                   confidence: float = 0.95) -> pd.DataFrame:
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    rows = []
    for col in reference.columns:
        n1, n2 = len(reference[col]), len(current[col])
        m1, m2 = reference[col].mean(), current[col].mean()
        s1, s2 = reference[col].std(ddof=1), current[col].std(ddof=1)
        diff = m2 - m1
        se = np.sqrt(s1**2 / n1 + s2**2 / n2)
        rows.append({
            "feature": col,
            "mean_diff": diff,
            "std_err": se,
            "ci_low": diff - z * se,
            "ci_high": diff + z * se,
        })
    return pd.DataFrame(rows).set_index("feature")


def flag_drifted_features(ks_results: pd.DataFrame, alpha: float = 0.05) -> list:
    sorted_results = ks_results.sort_values("p_value").reset_index()
    m = len(sorted_results)
    sorted_results["rank"] = np.arange(1, m + 1)
    sorted_results["bh_threshold"] = (sorted_results["rank"] / m) * alpha
    passing = sorted_results[sorted_results["p_value"] <= sorted_results["bh_threshold"]]
    if passing.empty:
        return []
    max_rank = passing["rank"].max()
    flagged = sorted_results[sorted_results["rank"] <= max_rank]
    return flagged["feature"].tolist()


summary = describe_batches(reference, current)
ks_results = ks_drift_test(reference, current)
ci_results = mean_diff_confidence_interval(reference, current)
drifted = flag_drifted_features(ks_results)

print(f"{len(drifted)} / {len(feature_names)} features flagged as drifted at alpha=0.05:")
print(drifted)
```

**Explanation.** Splitting on `mean radius` doesn't just shift `mean radius` itself — it drags
along every feature that's strongly correlated with tumor size: `mean perimeter`, `mean area`,
`worst radius`, `worst perimeter`, `worst area`, and (more weakly) their `se`-prefixed
standard-error counterparts. The KS test picks these up as large statistics with tiny p-values.
Features that are largely independent of size — `mean smoothness`, `mean symmetry`,
`mean fractal dimension`, `mean texture` — should show p-values close to uniform noise, and after
the Benjamini-Hochberg correction most of them fall out of the flagged set. This mirrors real
production drift: a single upstream cause (a new device, a new population segment) doesn't shift
every feature independently, it shifts a correlated cluster — and a monitoring system that just
eyeballs raw p-values per metric, without correcting for the number of features it's testing,
will drown in false alarms on the unrelated ones.

</details>